# Reaction Rates

MobsPy defaults to mass-action kinetics, but supports several rate function styles: constants, lambda functions, state-dependent functions, MobsPy expressions, and raw strings.

In [ ]:
from mobspy import *

## Constant Rates (Mass Action)

The simplest rate is a constant. MobsPy multiplies it by the reactant counts to form a mass-action rate law.

In [ ]:
A, B = BaseSpecies()

A >> B @ 0.5

sim = Simulation(A | B)
print(sim.compile())

## Lambda Functions

Lambda functions receive the reactant states as arguments. MobsPy calls the function once for each concrete reaction generated from the meta-reaction. Use `.is_a()` to check inheritance.

In [ ]:
Cell = BaseSpecies()
FastCell = New(Cell)
SlowCell = New(Cell)

# Fast cells degrade 10x faster than slow cells
Cell >> Zero @ (lambda c: 1.0 if c.is_a(FastCell) else 0.1)

sim2 = Simulation(FastCell | SlowCell)
print(sim2.compile())

## State-Dependent Rates

For species with characteristics, define a function that inspects the state and returns different rate constants.

In [ ]:
Bacterium = BaseSpecies()
Bacterium.healthy, Bacterium.sick


def death_rate(b):
    if b.healthy:
        return 0.01
    return 0.5


Bacterium >> Zero @ death_rate

sim3 = Simulation(Bacterium)
print(sim3.compile())

## Age-Dependent Duplication

You can combine inheritance queries with state queries. Here, old replicators duplicate faster than young ones.

In [ ]:
Age = BaseSpecies()
Age.young >> Age.old @ 1

Replicator = New(Age)

Replicator >> 2 * Replicator.young @ (lambda r: 2 if r.old else 1)

sim4 = Simulation(Replicator)
print(sim4.compile())

## MobsPy Expressions

MobsPy expressions let you write rates that reference species counts directly. The reactant arguments in a lambda become symbolic when used in arithmetic, producing a custom rate law instead of mass action.

In [ ]:
X, P = BaseSpecies()

Zero >> X @ 1
X >> X + P @ (lambda x: x / (1 + (10 / x) ** 4))

sim5 = Simulation(X | P)
print(sim5.compile())

## Michaelis-Menten-Like Rates

Expressions support full arithmetic, so you can build saturation kinetics directly.

In [ ]:
S, E, P2 = BaseSpecies()

# Enzyme-catalyzed conversion with saturation
S >> P2 @ (lambda s: 10 * s / (s + 50))

sim6 = Simulation(S | E | P2)
print(sim6.compile())

## String Rates

String rates are passed through to SBML without modification or validation. Use them when you need COPASI-specific syntax or want full manual control over the rate expression.

In [ ]:
M, N = BaseSpecies()

M >> N @ "M * 0.5 / (M + 100)"

sim7 = Simulation(M | N)
print(sim7.compile())

Note that string rates bypass all MobsPy checks. The string is inserted verbatim into the SBML kinetic law. Species names in the string must match the compiled species names exactly.